In [1]:
from pathlib import Path
import torch
import cv2
from typing import List

In [15]:
sap_path = Path("/home/stellasu/scratch/stellasu/sapien_poses/QVID/00000284.pt")
mp_path = Path("/home/stellasu/scratch/stellasu/poses/QVID/00000284.pt")

In [16]:
sap_ten = torch.load(sap_path, map_location="cpu")
mp_ten = torch.load(mp_path, map_location="cpu")

In [18]:
# sap_ten["bboxes"]
# sap_ten["bbox_scores"]
sap_ten["keypoints"]
# sap_ten["keypoints_visible"]
sap_ten["keypoint_scores"]
sap_ten["total_frames"]
sap_ten["video_path"]

'/orcd/compute/ppliang/001/long_video_datasets/short_video/QVID/videos/00000284.mp4'

In [31]:
merged_tensor = torch.cat([sap_ten["keypoints"], sap_ten["keypoint_scores"].unsqueeze(-1)], dim=-1)
merged_tensor = merged_tensor.flatten(start_dim=1)
print(merged_tensor.shape)

torch.Size([149, 798])


## Test Sampling

In [155]:
test = torch.load("/home/stellasu/scratch/stellasu/sapien_poses/HMDB51/brush_hair/Brushing_hair__the_right_way_brush_hair_u_nm_np1_fr_goo_0.perframe_fps1.00_max16.pt", map_location="cpu")
# test = torch.load("/home/stellasu/scratch/stellasu/poses/QVID/00000284.pt", map_location="cpu")
print("Pose Feat Min/Max/Mean:", test.min().item(), test.max().item(), test.mean().item())
# test2 = torch.load("/home/stellasu/scratch/stellasu/poses/QVID/00000284.perframe_fps8.00_max64.pt", map_location="cpu")

Pose Feat Min/Max/Mean: 0.0005973626975901425 1.3408045768737793 0.43686604499816895


In [99]:
# test = torch.load("/home/stellasu/scratch/stellasu/sapien_poses/HMDB51/eat/CastAway1_eat_u_nm_np1_fr_med_23.perframe_fps1.00_max16.pt", map_location="cpu")

print("Pose Feat Min/Max/Mean:", test.min().item(), test.max().item(), test.mean().item())

Pose Feat Min/Max/Mean: 0.0 253.14186096191406 46.63469696044922


In [117]:
test = torch.load("/home/stellasu/scratch/stellasu/sapien_poses/QVID/00000284.pt", map_location="cpu")
# print(test["total_frames"])
# print(test["keypoints"].shape)
# len(test["keypoints"])

keys = test["keypoints"]
scores = test["keypoint_scores"]

print("Pose Feat Min/Max/Mean:", keys.min().item(), keys.max().item(), keys.mean().item())

Pose Feat Min/Max/Mean: -94.80706787109375 652.327392578125 135.8238067626953


In [147]:
data = torch.load("/home/stellasu/scratch/stellasu/sapien_poses/QVID/00000284.pt", map_location="cpu")
video_path = "/home/stellasu/scratch/stellasu/long_video_datasets/short_video/QVID/videos/00000284.mp4"
cap = cv2.VideoCapture(video_path)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()

keypoints = data['keypoints']  # Shape: (N_frames, 2, 133, 2)
scores = data['keypoint_scores']  # Shape: (N_frames, 2, 133)

x_coords = keypoints[..., 0]
y_coords = keypoints[..., 1]
x_norm = x_coords / width
y_norm = y_coords / height
normalized_keypoints = torch.stack([x_norm, y_norm], dim=-1)

dummy_person = (scores.sum(dim=-1) == 0.0) 
dummy_person = dummy_person.unsqueeze(-1).unsqueeze(-1)
normalized_keypoints = torch.where(dummy_person, torch.tensor(-1.0), normalized_keypoints)

print("Original shape:", normalized_keypoints.shape)
print("Normalized Range Min/Max:", normalized_keypoints.min().item(), normalized_keypoints.max().item())

Original shape: torch.Size([149, 2, 133, 2])
Normalized Range Min/Max: -1.0 1.1173335313796997


In [49]:
video_path = "/home/stellasu/scratch/stellasu/long_video_datasets/short_video/QVID/videos/00000284.mp4"
fps = 1
max_frames = 16

cap = cv2.VideoCapture(video_path)
video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
stride = max(1.0, video_fps / fps)
indices: List[int] = []
pos = 0.0
while pos < total and len(indices) < max_frames:
    indices.append(min(round(pos), total - 1))
    pos += stride
if not indices:
    indices = [0]
indices = sorted(list(set(indices)))
cap.release()

In [131]:
video_path = "/home/stellasu/scratch/stellasu/long_video_datasets/short_video/QVID/videos/00000284.mp4"
cap = cv2.VideoCapture(video_path)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()

In [132]:
width, height

(640.0, 360.0)